### Save OPT-125M locally as a checkpoint

This snippet loads the pretrained OPT-125M model and tokenizer from Hugging Face, then writes a reusable local checkpoint:
- **Load artifacts**: Downloads model weights/config and tokenizer vocab/config for `facebook/opt-125m` (cached for reuse).
- **Save locally**: `model.save_pretrained('./opt125_checkpoint')` and `tokenizer.save_pretrained('./opt125_checkpoint')` create a folder with all files needed to reload offline.
- **Reload later**: You can load with `AutoModelForCausalLM.from_pretrained('./opt125_checkpoint')` and `AutoTokenizer.from_pretrained('./opt125_checkpoint')`.
- Note: This runs on CPU by default; move to GPU with `model.to('cuda')` if desired before generation (not required for saving).


In [ ]:
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# load model and tokenizer
model_name = "facebook/opt-125m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# save hf checkpoint under current directory (relative paths)
BASE_DIR = Path(".")
CKPT_DIR = BASE_DIR / "opt125_checkpoint"
model.save_pretrained(str(CKPT_DIR))
tokenizer.save_pretrained(str(CKPT_DIR))

print(f"Saved checkpoint to {CKPT_DIR}")

/home/rifatxia/Desktop/TensorstoreWork/tensorstore/opt-checkpointing/opt125/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved checkpoint to /home/rifatxia/Desktop/TensorstoreWork/tensorstore/opt-checkpointing/opt125_checkpoint


### Save weights via TensorStore (PyTorch state_dict → Tiled Zarr)

We mirror `save_pretrained` by exporting the model `state_dict()` and writing each tensor to a TensorStore-backed Zarr array. This yields a directory that can be reloaded offline without depending on Hugging Face save utilities.

- **Layout**: `./opt125_ts/weights/{param_name}` where each param is a chunked, compressed array.
- **Metadata**: We also write a minimal `config.json` alongside to mimic `save_pretrained`.
- **Reload**: Read Zarr arrays into CPU tensors and call `load_state_dict`.


In [ ]:
import os, json, math
from pathlib import Path
from typing import Dict

import torch
import tensorstore as ts

BASE_DIR = Path(".")
SAVE_DIR = BASE_DIR / "opt125_ts"
WEIGHTS_DIR = SAVE_DIR / "weights"

# dirs
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# state on cpu
state: Dict[str, torch.Tensor] = {k: v.detach().to("cpu") for k, v in model.state_dict().items()}

# simple chunk heuristic
def pick_chunks(t: torch.Tensor, target_bytes: int = 1_000_000):
    shape = list(t.shape)
    if len(shape) == 0:
        return None
    elem = t.element_size()
    chunks = shape.copy()
    while math.prod(chunks) * elem > target_bytes and chunks[-1] > 1:
        chunks[-1] = max(1, chunks[-1] // 2)
    return chunks

# write tensors
for name, tensor in state.items():
    arr_path = WEIGHTS_DIR / name
    arr_path.parent.mkdir(parents=True, exist_ok=True)

    np_arr = tensor.numpy()
    zarr_dtype = np_arr.dtype.newbyteorder('<').str

    metadata = {
        "dtype": zarr_dtype,
        "shape": list(np_arr.shape),
        "compressor": {"id": "zstd", "level": 3},
        "order": "C",
    }
    chunks = pick_chunks(tensor)
    if chunks is not None:
        metadata["chunks"] = chunks

    spec = {
        "driver": "zarr",
        "kvstore": {"driver": "file", "path": str(arr_path)},
        "metadata": metadata,
        "create": True,
        "delete_existing": True,
    }
    tstore = ts.open(spec).result()
    tstore.write(np_arr).result()

# minimal config
config = model.config.to_dict() if hasattr(model, "config") else {}
(SAVE_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"Saved TensorStore checkpoint to {SAVE_DIR}")


Saved TensorStore checkpoint to /home/rifatxia/Desktop/TensorstoreWork/tensorstore/opt-checkpointing/opt125_ts


### Load model weights back from TensorStore

This reads each Zarr array under `./opt125_ts/weights`, rebuilds a PyTorch `state_dict`, instantiates a model from `config.json`, and loads the weights. You can then run generation as usual.


In [ ]:
from transformers import AutoConfig

# create empty model from saved config
cfg = AutoConfig.from_pretrained("./opt125_ts")
reloaded = AutoModelForCausalLM.from_config(cfg)

# read tensors from tensorstore
reloaded_state = {}
for root, dirs, files in os.walk(WEIGHTS_DIR):
    if ".zarray" in files:
        zarr_root = root
        rel_root = os.path.relpath(zarr_root, WEIGHTS_DIR)
        spec = {"driver": "zarr", "kvstore": {"driver": "file", "path": zarr_root}}
        arr = ts.open(spec, read=True).result()
        np_arr = arr.read().result()
        key = rel_root.replace("\\", "/")
        reloaded_state[key] = torch.from_numpy(np_arr)

# load weights
missing, unexpected = reloaded.load_state_dict(reloaded_state, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

# quick generate and model identity check
pipe_inputs = tokenizer("Hello, my name is", return_tensors="pt")
with torch.no_grad():
    out1 = model.generate(**pipe_inputs, max_length=20)
    out2 = reloaded.generate(**pipe_inputs, max_length=20)
print("orig:", tokenizer.decode(out1[0], skip_special_tokens=True))
print("reloaded:", tokenizer.decode(out2[0], skip_special_tokens=True))

# hash of state_dict tensors to prove equality
import hashlib

def hash_state_dict(sd):
    h = hashlib.sha256()
    for k in sorted(sd.keys()):
        h.update(k.encode())
        h.update(sd[k].detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

orig_hash = hash_state_dict(model.state_dict())
reloaded_hash = hash_state_dict(reloaded.state_dict())
print("state_dict hash orig:", orig_hash)
print("state_dict hash reloaded:", reloaded_hash)
print("same:", orig_hash == reloaded_hash)


Missing keys: []
Unexpected keys: []
orig: Hello, my name is J.C. and I am a student at the University of California
reloaded: Hello, my name is J.C. and I am a student at the University of California
state_dict hash orig: 681529e645f341e1c148dd9f041c3884864df313f6314466377e1056cf2646ba
state_dict hash reloaded: 681529e645f341e1c148dd9f041c3884864df313f6314466377e1056cf2646ba
same: True


orig: Hello, my name is J.C. and I am a student at the University of California
reloaded: Hello, my name is Kari. I am a student of the art of photography. I


state_dict hash orig: 681529e645f341e1c148dd9f041c3884864df313f6314466377e1056cf2646ba
state_dict hash reloaded: 681529e645f341e1c148dd9f041c3884864df313f6314466377e1056cf2646ba
same: True
